In [ ]:
from google.colab import files
files.upload()

Saving train(1).parquet to train(1).parquet
Saving test(1).parquet to test(1).parquet
Saving previous_application(1).parquet to previous_application(1).parquet
Saving pos_cash(1).parquet to pos_cash(1).parquet
Saving installments(1).parquet to installments(1).parquet
Saving credit_balance.parquet to credit_balance.parquet
Saving bureau_agg_curr.parquet to bureau_agg_curr.parquet
Buffered data was truncated after reaching the output size limit.

In [ ]:
import pandas as pd
import numpy as np
!pip install shap
import shap
import sklearn
!pip install woodelf_explainer

In [ ]:
import zipfile

with zipfile.ZipFile("home_credit_features.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

BadZipFile: File is not a zip file

In [ ]:
import pandas as pd

bureau_agg_curr = pd.read_parquet("bureau_agg_curr.parquet")
credit_balance_agg = pd.read_parquet("credit_balance.parquet")

installment_payment_agg = pd.read_parquet("installments(1).parquet")
pos_cash_balance_agg = pd.read_parquet("pos_cash(1).parquet")
previous_application_final = pd.read_parquet("previous_application(1).parquet")

train = pd.read_parquet("train(1).parquet")
test = pd.read_parquet("test(1).parquet")

FileNotFoundError: [Errno 2] No such file or directory: 'credit_balance.parquet'

In [ ]:
for name, df in {
    "train": train,
    "bureau": bureau_agg_curr,
    "prev": previous_application_final,
    "cc": credit_balance_agg,
    "inst": installment_payment_agg,
    "pos": pos_cash_balance_agg
}.items():
    dup = df["SK_ID_CURR"].duplicated().sum()
    print(f"{name}: {dup}")

train: 0
bureau: 0
prev: 0
cc: 0
inst: 0
pos: 0


In [ ]:
train_final = train.copy()

train_final = train_final.merge(bureau_agg_curr, on="SK_ID_CURR", how="left")
train_final = train_final.merge(previous_application_final, on="SK_ID_CURR", how="left")
train_final = train_final.merge(credit_balance_agg, on="SK_ID_CURR", how="left")
train_final = train_final.merge(installment_payment_agg, on="SK_ID_CURR", how="left")
train_final = train_final.merge(pos_cash_balance_agg, on="SK_ID_CURR", how="left")

In [ ]:
test_final = test.copy()

test_final = test_final.merge(bureau_agg_curr, on="SK_ID_CURR", how="left")
test_final = test_final.merge(previous_application_final, on="SK_ID_CURR", how="left")
test_final = test_final.merge(credit_balance_agg, on="SK_ID_CURR", how="left")
test_final = test_final.merge(installment_payment_agg, on="SK_ID_CURR", how="left")
test_final = test_final.merge(pos_cash_balance_agg, on="SK_ID_CURR", how="left")

In [ ]:
print(train_final.shape)
print(test_final.shape)

print(train_final["SK_ID_CURR"].is_unique)

(307511, 199)
(48744, 198)
True


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.5 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

TARGET = "TARGET"

X = train_final.drop(columns=[TARGET])
y = train_final[TARGET]

In [ ]:
X = X.drop(columns=["SK_ID_CURR"], errors="ignore")

In [ ]:
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

for col in X.select_dtypes(include=["object"]).columns:
    X[col] = X[col].fillna("missing").astype(str)


In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n===== Fold {fold+1} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        eval_metric="AUC",
        random_seed=42,
        verbose=100
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        early_stopping_rounds=100
    )

    preds = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, preds)

    print(f"Fold {fold+1} ROC-AUC: {score:.5f}")
    scores.append(score)


===== Fold 1 =====
0:	test: 0.6500032	best: 0.6500032 (0)	total: 1.26s	remaining: 20m 58s
100:	test: 0.7534751	best: 0.7534751 (100)	total: 1m 55s	remaining: 17m 9s
200:	test: 0.7631131	best: 0.7631131 (200)	total: 3m 48s	remaining: 15m 9s
300:	test: 0.7683011	best: 0.7683011 (300)	total: 5m 43s	remaining: 13m 17s
400:	test: 0.7714864	best: 0.7714864 (400)	total: 7m 36s	remaining: 11m 21s
500:	test: 0.7743113	best: 0.7743113 (500)	total: 9m 31s	remaining: 9m 29s
600:	test: 0.7761524	best: 0.7761524 (600)	total: 11m 24s	remaining: 7m 34s
700:	test: 0.7774589	best: 0.7774589 (700)	total: 13m 17s	remaining: 5m 39s
800:	test: 0.7782932	best: 0.7782932 (800)	total: 15m 12s	remaining: 3m 46s
900:	test: 0.7789385	best: 0.7789468 (898)	total: 17m 5s	remaining: 1m 52s
999:	test: 0.7795721	best: 0.7795721 (999)	total: 18m 55s	remaining: 0us

bestTest = 0.7795720705
bestIteration = 999

Fold 1 ROC-AUC: 0.77957

===== Fold 2 =====
0:	test: 0.6449732	best: 0.6449732 (0)	total: 1.06s	remaining: 17m

In [ ]:
final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    eval_metric="AUC",
    random_seed=42,
    verbose=200
)

final_model.fit(X, y, cat_features=cat_features)

0:	total: 2.44s	remaining: 40m 37s
200:	total: 4m 37s	remaining: 18m 21s
400:	total: 9m 10s	remaining: 13m 41s
600:	total: 13m 36s	remaining: 9m 1s
800:	total: 17m 58s	remaining: 4m 28s
999:	total: 22m 27s	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=1000, learning_rate=0.03, random_seed=42, verbose=200)

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": final_model.get_feature_importance()
})

feature_importance = feature_importance.sort_values(
    by="importance", ascending=False
).reset_index(drop=True)

feature_importance.head(20)

,feature,importance
0,EXT_SOURCE_3,14.478802
1,EXT_SOURCE_2,14.440602
2,EXT_SOURCE_1,5.347880
3,DAYS_BIRTH,3.432473
4,debt_to_credit_ratio,2.851002
5,avg_latest_installments_left,2.591945
6,AMT_CREDIT,2.555230
7,AMT_GOODS_PRICE,2.336206
8,AMT_ANNUITY,2.127662
9,CODE_GENDER,2.114083


In [ ]:
final_model.score

<bound method CatBoostClassifier.score of CatBoostClassifier(depth=6, eval_metric='AUC', iterations=1000, learning_rate=0.03, random_seed=42, verbose=200)>

In [ ]:
for col in test_final.select_dtypes(include=["object"]).columns:
    test_final[col] = test_final[col].fillna("missing").astype(str)

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
test_preds = np.zeros(len(test_final))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n===== Fold {fold+1} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        eval_metric="AUC",
        random_seed=42,
        verbose=100
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        early_stopping_rounds=100
    )

    # 🔹 OOF predictions (train)
    oof[val_idx] = model.predict_proba(X_val)[:, 1]

    # 🔹 Test predictions (averaged)
    test_preds += model.predict_proba(test_final[X.columns])[:, 1] / kf.n_splits

    score = roc_auc_score(y_val, oof[val_idx])
    print(f"Fold {fold+1} ROC-AUC: {score:.5f}")

# =========================
# FINAL OOF SCORE
# =========================
print("\n========================")
print("OOF ROC-AUC:", roc_auc_score(y, oof))
print("========================")


===== Fold 1 =====
0:	test: 0.6500032	best: 0.6500032 (0)	total: 1.27s	remaining: 21m 5s
100:	test: 0.7534751	best: 0.7534751 (100)	total: 1m 59s	remaining: 17m 40s
200:	test: 0.7631131	best: 0.7631131 (200)	total: 3m 55s	remaining: 15m 36s
300:	test: 0.7683011	best: 0.7683011 (300)	total: 5m 51s	remaining: 13m 36s
400:	test: 0.7714864	best: 0.7714864 (400)	total: 7m 49s	remaining: 11m 41s
500:	test: 0.7743113	best: 0.7743113 (500)	total: 9m 47s	remaining: 9m 44s
600:	test: 0.7761524	best: 0.7761524 (600)	total: 11m 43s	remaining: 7m 47s
700:	test: 0.7774589	best: 0.7774589 (700)	total: 13m 38s	remaining: 5m 49s
800:	test: 0.7782932	best: 0.7782932 (800)	total: 15m 34s	remaining: 3m 52s
900:	test: 0.7789385	best: 0.7789468 (898)	total: 17m 33s	remaining: 1m 55s
999:	test: 0.7795721	best: 0.7795721 (999)	total: 19m 26s	remaining: 0us

bestTest = 0.7795720705
bestIteration = 999

Fold 1 ROC-AUC: 0.77957

===== Fold 2 =====
0:	test: 0.6449732	best: 0.6449732 (0)	total: 1.06s	remaining: 1

In [ ]:
submission = pd.DataFrame({
    "SK_ID_CURR": test["SK_ID_CURR"],
    "TARGET": test_preds
})

submission.to_csv("submission.csv", index=False)